In [32]:
from openai import OpenAI
import os
import json
import subprocess
import gradio as gr
Workspace = "workspace"

In [33]:
#llm init in agent
def model_init(api_key,base_url)->object:
    model=OpenAI(api_key=api_key,base_url=base_url)
    return model

In [34]:
#tools
def read_file(path)->str:
    path=os.path.join(Workspace,path)
    with open(path,"r") as file:
        text=file.read()
    return text
    
def write_file(path,code)->None:
    path=os.path.join(Workspace,path)
    with open(path,"w") as file:
        file.write(code)

def list_files(path=".")->list:
    path=os.path.join(Workspace,path)
    return os.listdir(path)

def run_command(command)->dict:
    try:
        result= subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        cwd=Workspace
        )
        return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "returncode": result.returncode
        }
    except Exception as e:
        return {
        "stdout": "",
        "stderr": str(e),
        "returncode": 1
    }

In [35]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to read."
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write code or text to a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to write."
                    },
                    "code": {
                        "type": "string",
                        "description": "The code or text to write into the file."
                    }
                },
                "required": ["path", "code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files and folders inside a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The directory path to list. Defaults to the current directory."
                    }
                },
                "required": []
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "run_command",
        "description": "Run a shell command and return its output.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to execute."
                }
            },
            "required": ["command"]
        }
    }
  }
]

In [36]:
def get_tool(tool_name,args):
    tool_dic={
        "read_file":read_file,
        "write_file":write_file,
        "list_files":list_files,
        "run_command":run_command
    }

    return tool_dic[tool_name](**args)

def tool_handler(msg):
    responses=[]
    for tool_call in msg.tool_calls:
        tool_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
        result=get_tool(tool_name=tool_name,args=args)

        responses.append({
            "role":"tool",
            "content":json.dumps(result),
            "tool_call_id":tool_call.id
        })
    return responses

In [37]:
def message():
    sys_msg="""you are a coding agent.
    you modify the code according to the user request.
    before modifying a file, read the file first.
    use the available tools when you need to read and modify a file or when you need to test the code.
    check if the targeted language compiler or interpreter is installed first, if its not installed don't test the commands in the run_command tool.
    if an error accures you inspect the error and fix it.
    keep your words simple and precise"""

    msg=[{"role":"system","content":sys_msg}]
    return msg

In [38]:
def coder(messages,model,model_name,reasoning="low"):
    
    response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

    while True:
        msg=response.choices[0].message
        if not msg.tool_calls:
            return msg.content
        messages.append(msg)
        tool_responses=tool_handler(msg)
        messages.extend(tool_responses)

        response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

     

In [39]:
model=model_init("gmm","http://localhost:11434/v1")

In [40]:
def chat(msg,history)->dict:
    messages=message()

    history=[{"role":h["role"],"content":h["content"]} for h in history]

    messages=messages+history+[{"role":"user","content":msg}]
    response= coder(messages,model,model_name="gemma4:e2b")
    
    history.append({"role": "user", "content": msg})
    history.append({"role": "assistant", "content": response})

    return history

def interface(workspace=".")->None:
    with gr.Blocks() as app:

        with gr.Row():

            with gr.Column(scale=1):
                gr.Markdown("Files")

                files=gr.FileExplorer(
                    glob="**/*",
                    root_dir=workspace,
                    file_count="multiple",
                    label="Project"
            )
            
            with gr.Column(scale=3):
                chatbot=gr.Chatbot(
                    label="agent"
            )

                message=gr.Textbox(
                    placeholder="type the file and the change you want in it",
            )

                send=gr.Button("send")

        send.click(
            chat,
            inputs=[message,chatbot],
            outputs=chatbot
        )

        message.submit(
            chat,
            inputs=[message,chatbot],
            outputs=chatbot
        )
    app.launch()

In [ ]:

interface(Workspace)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
